# Task 4: Build a Mini Traffic Analytics Application

In [1]:
import logging

log_file = r"C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\part3_machine_learning\unsupervised_ML_models.log"

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

logger.handlers.clear()

file_handler = logging.FileHandler(
    log_file,
    mode="w",
    encoding="utf-8"
)

formatter = logging.Formatter(
    "{asctime} - {levelname} - {name} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Test that logging is working
logger.info("Logging started successfully.")

In [3]:
import pandas as pd
import numpy as np

NameError: name 'X' is not defined

In [3]:
# Load processed traffic dataset
def loadDataBase():

    global DataBase

    file_path = r"C:\\Users\\symev\\OneDrive\\Training\\Emeritus\\capstone project\\emeritus captone project\\Metro_Interstate_Traffic_Volume_Cleaned_FeatureEngineering.csv"

    try:
        # Load CSV
        DataBase = pd.read_csv(file_path)

        # Convert date_time to datetime
        DataBase['date_time'] = pd.to_datetime(
            DataBase['date_time'],
            errors='coerce'
        )

        print("\nDatabase loaded successfully.")
        print(f"Rows: {len(DataBase)}")
        print(f"Columns: {len(DataBase.columns)}")

        # Confirm date_time conversion
        print(f"date_time data type: {DataBase['date_time'].dtype}")

    except FileNotFoundError:
        print(f"Error: File not found: {file_path}")

    except pd.errors.EmptyDataError:
        print("Error: The CSV file is empty.")

    except pd.errors.ParserError:
        print("Error: The CSV file could not be parsed.")

    except OSError as e:
        print(f"Error reading the file: {e}")

In [4]:
loadDataBase()


Database loaded successfully.
Rows: 48176
Columns: 30
date_time data type: datetime64[ns]


In [5]:
# Query traffic data by date
def query_traffic():

    # Ask user for date
    date_input = input("Enter the date to query (YYYY-MM-DD): ")

    # Log the command and argument
    logger.info(
        "Command invoked: querytraffic | Argument: date=%s",
        date_input
    )

    try:
        # Convert user input to datetime
        query_date = pd.to_datetime(
            date_input,
            format="%Y-%m-%d"
        ).date()

    except ValueError:
        # Log clear error instead of showing traceback
        logger.error(
            "Invalid input for querytraffic. "
            "Malformed date supplied: '%s'. "
            "Expected format: YYYY-MM-DD.",
            date_input
        )

        print(
            "Error: Invalid date. "
            "Please enter the date in YYYY-MM-DD format."
        )
        return

    # Filter dataset by selected date
    results = DataBase[
        DataBase['date_time'].dt.date == query_date
    ]

    # Check whether data exists for the selected date
    if results.empty:

        logger.info(
            "Command completed: querytraffic | "
            "No records found for date=%s",
            query_date
        )

        print(f"\nNo traffic data found for {query_date}.")
        return

    # Calculate most common weather condition
    weather_mode = results['weather_main'].mode()

    # Calculate average traffic volume
    average_traffic = results['traffic_volume'].mean()

    # Display results
    print(f"\nTraffic analysis for {query_date}")
    print("-----------------------------------")

    if not weather_mode.empty:
        print(
            f"Most common weather condition: "
            f"{weather_mode.iloc[0]}"
        )
    else:
        print("Most common weather condition: No data")

    print(f"Average traffic volume: {average_traffic:.2f}")
    print(f"Number of records: {len(results)}")

    # Log successful command
    logger.info(
        "Command completed successfully: querytraffic | "
        "Date=%s | Records=%d | Weather mode=%s | "
        "Average traffic volume=%.2f",
        query_date,
        len(results),
        weather_mode.iloc[0] if not weather_mode.empty else "No data",
        average_traffic
    )


In [6]:
query_traffic()

Enter the date to query (YYYY-MM-DD): 

No traffic data found for NaT.


In [7]:
# Inform user of likely congestion level of traffic based on chosen weekday/weekend and hour
def query_congestion():

    # Ask whether the user is travelling on a weekday or weekend
    day_type = input(
        "Are you travelling on a weekday or weekend? "
        "(weekday/weekend): "
    ).strip().lower()

    # Ask for intended travel hour
    time_input = input(
        "What hour do you intend to travel? (00-23): "
    ).strip()

    # Log command and arguments
    logger.info(
        "Command invoked: query_congestion | "
        "Arguments: day_type=%s, hour=%s",
        day_type,
        time_input
    )

    # Validate weekday/weekend input
    if day_type not in ["weekday", "weekend"]:

        logger.error(
            "Invalid input for query_congestion. "
            "Expected 'weekday' or 'weekend', received '%s'.",
            day_type
        )

        print(
            "Error: Please enter either 'weekday' or 'weekend'."
        )
        return

    # Validate hour input
    try:
        hour = int(time_input)

        if hour < 0 or hour > 23:
            raise ValueError

    except ValueError:

        logger.error(
            "Invalid input for query_congestion. "
            "Invalid hour '%s'. Expected an integer between 0 and 23.",
            time_input
        )

        print(
            "Error: Please enter an hour between 0 and 23."
        )
        return

    # Filter based on weekday/weekend
    if day_type == "weekend":
        filtered_data = DataBase[
            (DataBase['is_weekend'] == 1) &
            (DataBase['hour'] == hour)
        ]

    else:
        filtered_data = DataBase[
            (DataBase['is_weekend'] == 0) &
            (DataBase['hour'] == hour)
        ]

    # Check whether matching records exist
    if filtered_data.empty:

        logger.info(
            "Command completed: query_congestion | "
            "No records found for %s at hour %02d.",
            day_type,
            hour
        )

        print(
            f"\nNo traffic data found for a {day_type} "
            f"at {hour:02d}:00."
        )
        return

    # Calculate mode of congestion category
    congestion_mode = filtered_data['congestion_category'].mode()

    # Display result
    print(
        f"\nTraffic prediction for {day_type} at {hour:02d}:00"
    )
    print("-----------------------------------------------")

    if not congestion_mode.empty:
        print(
            f"Most likely congestion category: "
            f"{congestion_mode.iloc[0]}"
        )
    else:
        print("Congestion category: No data")

    print(f"Number of matching records: {len(filtered_data)}")

    # Log successful query
    logger.info(
        "Command completed successfully: query_congestion | "
        "Day type=%s | Hour=%02d | Records=%d | "
        "Congestion mode=%s",
        day_type,
        hour,
        len(filtered_data),
        congestion_mode.iloc[0] if not congestion_mode.empty else "No data"
    )

In [8]:
query_congestion()

Are you travelling on a weekday or weekend? (weekday/weekend): 
What hour do you intend to travel? (00-23): 
Error: Please enter either 'weekday' or 'weekend'.


In [9]:
# Query traffic by weather condition and weekday/weekend
def query_weather_day():

    # Display available weather conditions
    weather_conditions = sorted(
        DataBase['weather_main'].dropna().unique()
    )

    print("\nAvailable weather conditions:")
    for weather in weather_conditions:
        print(f"- {weather}")

    # Ask user for weather condition
    weather_input = input(
        "\nSelect a weather condition: "
    ).strip()

    # Ask user for weekday/weekend
    day_type = input(
        "Are you travelling on a weekday or weekend? "
        "(weekday/weekend): "
    ).strip().lower()

    # Log command and arguments
    logger.info(
        "Command invoked: query_weather_day | "
        "Arguments: weather=%s, day_type=%s",
        weather_input,
        day_type
    )

    # Validate weather condition
    matching_weather = [
        weather for weather in weather_conditions
        if weather.lower() == weather_input.lower()
    ]

    if not matching_weather:

        logger.error(
            "Invalid input for query_weather_day. "
            "Unknown weather condition: '%s'.",
            weather_input
        )

        print(
            f"Error: '{weather_input}' is not a valid weather condition."
        )
        return

    # Use the correctly formatted weather value
    weather = matching_weather[0]

    # Validate weekday/weekend
    if day_type not in ["weekday", "weekend"]:

        logger.error(
            "Invalid input for query_weather_day. "
            "Expected 'weekday' or 'weekend', received '%s'.",
            day_type
        )

        print(
            "Error: Please enter either 'weekday' or 'weekend'."
        )
        return

    # Filter the dataset
    if day_type == "weekend":

        filtered_data = DataBase[
            (DataBase['weather_main'] == weather) &
            (DataBase['is_weekend'] == 1)
        ]

    else:

        filtered_data = DataBase[
            (DataBase['weather_main'] == weather) &
            (DataBase['is_weekend'] == 0)
        ]

    # Check if matching records exist
    if filtered_data.empty:

        logger.info(
            "Command completed: query_weather_day | "
            "No records found for weather=%s and day_type=%s.",
            weather,
            day_type
        )

        print(
            f"\nNo data found for {weather} weather "
            f"on a {day_type}."
        )
        return

    # Calculate average traffic volume
    average_traffic = filtered_data['traffic_volume'].mean()

    # Calculate mode congestion category
    congestion_mode = filtered_data['congestion_category'].mode()

    # Display results
    print("\nTraffic Analysis")
    print("-----------------------------------")
    print(f"Weather condition: {weather}")
    print(f"Day type: {day_type.capitalize()}")
    print(f"Average traffic volume: {average_traffic:.2f}")

    if not congestion_mode.empty:
        print(
            f"Most common congestion category: "
            f"{congestion_mode.iloc[0]}"
        )
    else:
        print("Most common congestion category: No data")

    print(f"Number of matching records: {len(filtered_data)}")

    # Log successful query
    logger.info(
        "Command completed successfully: query_weather_day | "
        "Weather=%s | Day type=%s | Records=%d | "
        "Average traffic volume=%.2f | Congestion mode=%s",
        weather,
        day_type,
        len(filtered_data),
        average_traffic,
        congestion_mode.iloc[0] if not congestion_mode.empty else "No data"
    )


In [10]:
query_weather_day()


Available weather conditions:
- Clear
- Clouds
- Drizzle
- Fog
- Haze
- Mist
- Rain
- Smoke
- Snow
- Squall
- Thunderstorm

Select a weather condition: 
Are you travelling on a weekday or weekend? (weekday/weekend): 
Error: '' is not a valid weather condition.


In [11]:
# Display menu and get choice
def menu():
    print("\n========================================")
    print("       SMART CITY TRAFFIC ANALYSIS")
    print("========================================")
    print("A. Query traffic by date")
    print("B. Query congestion by day and hour")
    print("C. Query traffic by weather and day type")
    print("D. Display database")
    print("Q. Quit application")
    print("========================================")

    choice = input("SELECT CHOICE: ").strip().upper()

    print("\n")

    return choice

In [12]:
menu()


       SMART CITY TRAFFIC ANALYSIS
A. Query traffic by date
B. Query congestion by day and hour
C. Query traffic by weather and day type
D. Display database
Q. Quit application
SELECT CHOICE: 




''

In [13]:
# Main application

# Load processed traffic dataset
loadDataBase()

# Display menu and dispatch user's choice
selected = ""

while selected != "Q":

    selected = menu()

    # Log the command selected by the user
    logger.info(
        "Command invoked: %s",
        selected
    )

    if selected == "A":
        query_traffic()

    elif selected == "B":
        query_congestion()

    elif selected == "C":
        query_weather_day()

    elif selected == "D":
        logger.info("Command invoked: display_database")

        print("\nDATABASE")
        print(DataBase)

    elif selected == "Q":
        logger.info("Command invoked: quit")
        print("Application closed.")

    else:
        # Log invalid menu choice as an ERROR
        logger.error(
            "Invalid menu choice supplied: '%s'. "
            "Expected A, B, C, D or Q.",
            selected
        )

        print(
            "Invalid choice. Please select A, B, C, D or Q."
        )


Database loaded successfully.
Rows: 48176
Columns: 30
date_time data type: datetime64[ns]

       SMART CITY TRAFFIC ANALYSIS
A. Query traffic by date
B. Query congestion by day and hour
C. Query traffic by weather and day type
D. Display database
Q. Quit application
SELECT CHOICE: A


Enter the date to query (YYYY-MM-DD): 2014-09-28

No traffic data found for 2014-09-28.

       SMART CITY TRAFFIC ANALYSIS
A. Query traffic by date
B. Query congestion by day and hour
C. Query traffic by weather and day type
D. Display database
Q. Quit application
SELECT CHOICE: a


Enter the date to query (YYYY-MM-DD): 2016-09-29

Traffic analysis for 2016-09-29
-----------------------------------
Most common weather condition: Clear
Average traffic volume: 3467.67
Number of records: 24

       SMART CITY TRAFFIC ANALYSIS
A. Query traffic by date
B. Query congestion by day and hour
C. Query traffic by weather and day type
D. Display database
Q. Quit application
SELECT CHOICE: a


Enter the date to que

SELECT CHOICE: q


Application closed.
